In [ ]:
import os
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import json
import requests
from IPython.display import clear_output, display
import sqlite3
import random
from datetime import datetime, timedelta


logged_in_user_id = None


def log_access(event_description, user_id=None, log_type="General"):
    """
    Logs an access event into the AccessLogs table.

    Parameters:
        event_description (str): A description of the event being logged.
        user_id (int, optional): The ID of the user associated with the event. Defaults to None.
        log_type (str): The type of log event (e.g., "General", "Login", "Update"). Defaults to "General".
    """
    db_path = os.path.join(os.getcwd(), "VegasIQ.db")
    
    if not os.path.exists(db_path):
        print("Database not found. Please initialize the system first.")
        return

    try:
        conn = sqlite3.connect(db_path)
        cursor = conn.cursor()

        # Ensure the LogType exists in LogTypeDimension
        cursor.execute("""
            INSERT OR IGNORE INTO LogTypeDimension (LogType, LogTypeDescription, FirstSeenDate, LastSeenDate, errorCount)
            VALUES (?, ?, DATE('now'), DATE('now'), 0)
        """, (log_type, event_description))

        # Update the LastSeenDate for the log type
        cursor.execute("""
            UPDATE LogTypeDimension
            SET LastSeenDate = DATE('now')
            WHERE LogType = ?
        """, (log_type,))

        # Retrieve the LogTypeID
        cursor.execute("""
            SELECT LogTypeID FROM LogTypeDimension WHERE LogType = ?
        """, (log_type,))
        log_type_id = cursor.fetchone()[0]

        # Insert the log into AccessLogs
        cursor.execute("""
            INSERT INTO AccessLogs (UserID, AccessDateTime, LogTypeID, create_date, update_date)
            VALUES (?, DATETIME('now'), ?, DATE('now'), DATE('now'))
        """, (user_id, log_type_id))

        conn.commit()

    except sqlite3.Error as e:
        print(f"Database error: {e}")
    except Exception as e:
        print(f"Unexpected error: {e}")
    finally:
        if conn:
            conn.close()
        

def authenticate_user():
    """
    Displays the authentication splash screen and validates the user's credentials.
    Sets the global `logged_in_user_id` if authentication is successful.
    Returns the permission level of the user if authentication is successful, otherwise None.
    """
    global logged_in_user_id
    db_path = os.path.join(os.getcwd(), "VegasIQ.db")
    
    if not os.path.exists(db_path):
        print("Database not found. Please initialize the system first.")
        return None

    try:
        conn = sqlite3.connect(db_path)
        cursor = conn.cursor()

        print("Welcome to VegasIQ - User Authentication")
        print("========================================")

        while True:
            username = input("Enter your username: ").strip()
            if not username:
                print("Username cannot be empty. Please try again.")
                continue

            password = input("Enter your password: ").strip()
            if not password:
                print("Password cannot be empty. Please try again.")
                continue
            break
        
        # Query the database to check for matching credentials and get the permission level
        cursor.execute("""
            SELECT UserID, permission_level 
            FROM UserTable 
            WHERE username = ? AND password = ?
        """, (username, password))

        user = cursor.fetchone()

        if user:
            logged_in_user_id = user[0]  # Store the UserID in the global variable
            permission_level = user[1]
            log_access(event_description="User has logged in to the system", log_type="System Log in", user_id=logged_in_user_id)
            print("Authentication successful!")
            input("Press Enter to continue to the main menu...")
            return permission_level
        else:
            print("Invalid username or password. Please try again.")
            input("Press Enter to retry...")
            log_access(event_description="Failed Authentication", log_type="System Log-in: Failed")
            return None

    except sqlite3.Error as e:
        print(f"An error occurred while accessing the database: {e}")
        return None

    finally:
        if 'conn' in locals() and conn:
            conn.close()




# Function to initialize the database
def initialize_database_once():
    """
    Initializes the VegasIQ.db SQLite database with necessary tables and sample data.
    Ensures that initialization happens only if the database file doesn't already exist.
    """
    db_path = os.path.join(os.getcwd(), "VegasIQ.db")
    if not os.path.exists(db_path):
        try:
            conn = sqlite3.connect(db_path)
            cursor = conn.cursor()

            # Create the UserTable
            cursor.execute("""
                CREATE TABLE IF NOT EXISTS UserTable (
                    UserID INTEGER PRIMARY KEY AUTOINCREMENT,
                    first_name TEXT NOT NULL,
                    last_name TEXT NOT NULL,
                    username TEXT NOT NULL UNIQUE,
                    password TEXT NOT NULL,
                    permission_level INTEGER NOT NULL CHECK(permission_level IN (1, 2)),
                    create_date DATE NOT NULL,
                    update_date DATE NOT NULL
                )
            """)

            # Create the AccessLogs table
            cursor.execute("""
                CREATE TABLE IF NOT EXISTS AccessLogs (
                    UserID INTEGER NULL,
                    AccessDateTime DATETIME NOT NULL,
                    LogTypeID INTEGER NOT NULL,
                    create_date DATE NOT NULL,
                    update_date DATE NOT NULL,
                    FOREIGN KEY(UserID) REFERENCES UserTable(UserID),
                    FOREIGN KEY(LogTypeID) REFERENCES LogTypeDimension(LogTypeID)
                )
            """)

            # Create the LogTypeDimension table
            cursor.execute("""
                CREATE TABLE IF NOT EXISTS LogTypeDimension (
                    LogTypeID INTEGER PRIMARY KEY AUTOINCREMENT,
                    LogType TEXT NOT NULL,
                    LogTypeDescription TEXT NOT NULL,
                    FirstSeenDate DATE NOT NULL,
                    LastSeenDate DATE NOT NULL,
                    errorCount INTEGER NOT NULL
                )
            """)

            # Populate the UserTable with 200 rows of sample data
            first_names = ["Jorge", "Keith", "Bereniz", "Ola", "Quan"]
            last_names = ["Alvarez", "Webster", "Castaneda", "Nguyen", "Balogun"]
            now = datetime.now()

            # Set to track unique usernames
            generated_usernames = set()

            # Generate and insert user data
            user_data = []
            for _ in range(200):
                while True:  # Loop until a unique username is generated
                    first_name = random.choice(first_names)
                    last_name = random.choice(last_names)
                    username = f"{first_name.lower()}.{last_name.lower()}{random.randint(1, 100)}"
                    if username not in generated_usernames:  # Check for uniqueness
                        generated_usernames.add(username)  # Add to the set of generated usernames
                        break
                
                password = "DefaultPassword"
                permission_level = random.choice([1, 2])
                create_date = (now - timedelta(days=random.randint(0, 365))).strftime("%Y-%m-%d")
                update_date = create_date
                
                # Append data to the list
                user_data.append((first_name, last_name, username, password, permission_level, create_date, update_date))

            cursor.executemany("""
            INSERT INTO UserTable (first_name, last_name, username, password, permission_level, create_date, update_date)
            VALUES (?, ?, ?, ?, ?, ?, ?)
            """, user_data)

            # Populate the UserTable with predefined users
            predefined_users = [
                ("Root", "Admin", "root", "admin", 2),  # Administrator user
                ("Regular", "User", "user", "password", 1)  # Regular user
            ]

            now = datetime.now().strftime("%Y-%m-%d")
            for first_name, last_name, username, password, permission_level in predefined_users:
                cursor.execute("""
                    INSERT INTO UserTable (first_name, last_name, username, password, permission_level, create_date, update_date)
                    VALUES (?, ?, ?, ?, ?, ?, ?)
                """, (first_name, last_name, username, password, permission_level, now, now))


            # commit to database
            conn.commit()
            log_access(event_description=f"Database initialized and populated successfully at {db_path}.", log_type="Database Init", user_id=logged_in_user_id)

        except sqlite3.Error as e:
            print(f"An error occurred while interacting with the database: {e}")

        except Exception as ex:
            print(f"An unexpected error occurred: {ex}")

        finally:
            if 'conn' in locals() and conn:
                conn.close()
    else:
        print("Exisiting Databse found. Skipping initialization.")
        log_access(event_description="Exisiting Databse found. Skipping initialization.", log_type="Database Init - Existing")


# Main menu display function
def display_main_menu(permission_level):
    """
    Displays the main menu dynamically based on the user's permission level.
    """
    print("Welcome to VegasIQ")
    print("===================")
    print("1. Las Vegas Events and Rooms Database")
    print("2. Las Vegas Weather Data")
    print("3. Las Vegas Airport Statistics")
    print("4. Clark County Meeting Space Inventory")
    if permission_level == 2:  # Show "User Management" only for level 2 users
        print("5. User Management")
    print("6. System Exit")
    try:
        log_access(event_description="User Displayed Main Menu", log_type="Display- Main Menu", user_id=logged_in_user_id)
        display("===================")
        return int(input("Select an option: "))
        
    except ValueError:
        return -1  # Return an invalid option to trigger error handling   return -1  # Return an invalid option to trigger error handlin


def user_management_menu():
    """
    Displays the user management menu and allows the admin to choose an action.
    """
    while True:
        clear_output(wait=True)
        print("User Management Menu")
        print("====================")
        print("1. Create a New User")
        print("2. Delete a User")
        print("3. Search for Users by Partial Username")
        print("4. Update Password by Partial Username Search")
        print("5. Update Permission Level by Partial Username Search")
        print("6. Return to Main Menu")
        print("====================")
        try:
            log_access(event_description="User Displayed 'User Management' Sub-Menu", user_id=logged_in_user_id, log_type="Display- User Management Menu")
            display("")
            return int(input("Select an option: "))
        except ValueError:
            print("Invalid input. Please try again.")
            continue


def search_users_by_partial_username(cursor, partial_username):
    """
    Searches for users by a partial username and handles cases with more than 10 results.
    Returns a list of users to the caller.
    """
    
    while True:
        cursor.execute("""
            SELECT UserID, username, first_name, last_name, permission_level
            FROM UserTable
            WHERE username LIKE ?
        """, (f"%{partial_username}%",))

        results = cursor.fetchall()
        log_access(event_description="User Table search for for partial username string", user_id=logged_in_user_id, log_type=f"Database- UserTable search username LIKE {partial_username}")

        if len(results) > 10:
            print(f"More than 10 users found ({len(results)}). Please refine your search or type 'quit' to go back.")
            partial_username = input("Enter a more specific search string: ").strip()
            if partial_username.lower() == 'quit':
                print("Exiting search.")
                return []
        elif len(results) == 0:
            print("No users found. Please try again or type 'quit' to go back.")
            partial_username = input("Enter a search string: ").strip()
            if partial_username.lower() == 'quit':
                print("Exiting search.")
                return []
        else:
            print("\nUsers Found:")
            for idx, (user_id, username, first_name, last_name, permission_level) in enumerate(results, 1):
                print(f"{idx}. UserID: {user_id}, Username: {username}, Name: {first_name} {last_name}, Permission Level: {permission_level}")
            return results



def create_user(cursor):
    """
    Creates a new user in the database.
    """
    while True:
        first_name = input("Enter first name: ").strip()
        if not first_name.isalpha():
            print("First name should contain only letters. Please try again.")
            continue
        break

    while True:
        last_name = input("Enter last name: ").strip()
        if not last_name.isalpha():
            print("Last name should contain only letters. Please try again.")
            continue
        break

    while True:
        username = input("Enter username: ").strip()
        if not username:
            print("Username cannot be empty. Please try again.")
            continue
        break
    
    while True:
        password = input("Enter password: ").strip()
        if (len(password) < 8 or not any(char.isdigit() for char in password) or
                not any(char.isupper() for char in password) or not any(char.islower() for char in password) or
                not any(char in "!@#$%^&*()-_+=<>?" for char in password)):
            print("Password must be at least 8 characters long and include an uppercase letter, a lowercase letter, a number, and a special character. Please try again.")
            continue
        confirm_password = input("Re-enter the new password for confirmation: ").strip()
        if password != confirm_password:
            print("Passwords do not match. Please try again.")
            continue
        break

    while True:
        permission_level = input("Enter permission level (1 for regular user, 2 for admin): ").strip()
        if permission_level not in ('1', '2'):
            print("Permission level must be either 1 or 2. Please try again.")
            continue
        break

    try:
        cursor.execute("""
            INSERT INTO UserTable (first_name, last_name, username, password, permission_level, create_date, update_date)
            VALUES (?, ?, ?, ?, ?, DATE('now'), DATE('now'))
        """, (first_name, last_name, username, password, int(permission_level)))
        print("User created successfully.")
        #log_access(event_description="Admin added a new user", user_id=logged_in_user_id, log_type="Database-  added to User table")
    except sqlite3.Error as e:
        print(f"Error creating user: {e}")
        #log_access(event_description="Failed to Add user to SQLite error", user_id=logged_in_user_id, log_type=f"Database ERROR- {e}")



def delete_user(cursor):
    """
    Deletes a user from the database.
    """
    partial_username = input("Enter a partial username to search for: ").strip()
    users = search_users_by_partial_username(cursor, partial_username)

    if users:
        while True:
            try:
                choice = int(input("Select a user to delete (enter the number, or 0 to go back): ")) - 1
                if choice == -1:
                    print("Going back to the previous menu.")
                    return
                if 0 <= choice < len(users):
                    break
                else:
                    print(f"Please enter a number between 1 and {len(users)}.")
            except ValueError:
                print("Invalid input. Please enter a valid number.")
        
        user_id = users[choice][0]
        
        confirm = input(f"Are you sure you want to delete user {users[choice][1]}? (yes/no): ").strip().lower()
        if confirm == 'yes':
            cursor.execute("DELETE FROM UserTable WHERE UserID = ?", (user_id,))
            print("User deleted successfully.")
            #log_access(event_description="Admin added a deleted user", user_id=logged_in_user_id, log_type=f"Database- UserID {user_id} deleted from user table")
        else:
            print("User deletion cancelled.")





def update_password(cursor):
    """
    Updates a user's password.
    """
    partial_username = input("Enter a partial username to search for: ").strip()
    users = search_users_by_partial_username(cursor, partial_username)

    if users:
        while True:
            try:
                choice = int(input("Select a user to update password (enter the number, or press 0 to go back): ")) - 1
                if choice == -1:
                    print("Going back to the previous menu.")
                    return
                if 0 <= choice < len(users):
                    break
                else:
                    print(f"Please enter a number between 1 and {len(users)}.")
            except ValueError:
                print("Invalid input. Please enter a valid number.")
        
        user_id = users[choice][0]
        
        while True:
            new_password = input("Enter the new password: ").strip()
            if (len(new_password) < 8 or not any(char.isdigit() for char in new_password) or
                    not any(char.isupper() for char in new_password) or not any(char.islower() for char in new_password) or
                    not any(char in "!@#$%^&*()-_+=<>?" for char in new_password)):
                print("Password must be at least 8 characters long and include an uppercase letter, a lowercase letter, a number, and a special character. Please try again.")
                continue
            confirm_password = input("Re-enter the new password for confirmation: ").strip()
            if new_password != confirm_password:
                print("Passwords do not match. Please try again.")
                continue
            break
        
        cursor.execute("UPDATE UserTable SET password = ?, update_date = DATE('now') WHERE UserID = ?", (new_password, user_id))
        print("Password updated successfully.")
        #log_access(event_description="Admin changed a user password", user_id=logged_in_user_id, log_type=f"Database- password changed for userID: {user_id}")

def update_permission_level(cursor):
    """
    Updates a user's permission level.
    """
    partial_username = input("Enter a partial username to search for: ").strip()
    users = search_users_by_partial_username(cursor, partial_username)

    if users:
        while True:
            try:
                choice = int(input("Select a user to update permission level (enter the number, or press 0 to go back): ")) - 1
                if choice == -1:
                    print("Going back to the previous menu.")
                    return
                if 0 <= choice < len(users):
                    break
                else:
                    print(f"Please enter a number between 1 and {len(users)}.")
            except ValueError:
                print("Invalid input. Please enter a valid number.")
        
        user_id = users[choice][0]
        
        while True:
            new_permission_level = input("Enter the new permission level (1 for regular user, 2 for admin): ").strip()
            if new_permission_level not in ('1', '2'):
                print("Permission level must be either 1 or 2. Please try again.")
                continue
            break
        
        cursor.execute("UPDATE UserTable SET permission_level = ?, update_date = DATE('now') WHERE UserID = ?", (int(new_permission_level), user_id))
        print("Permission level updated successfully.")
        #log_access(event_description="Admin changed user permission level", user_id=logged_in_user_id, log_type=f"Database- userID: {user_id} permission level changed to {new_permission_level}")



def user_management():
    """
    Main function for user management.
    """
    db_path = os.path.join(os.getcwd(), "VegasIQ.db")
    if not os.path.exists(db_path):
        print("Database not found. Please initialize the system first.")
        return

    try:
        conn = sqlite3.connect(db_path)
        cursor = conn.cursor()

        while True:
            choice = user_management_menu()
            if choice == 1:
                create_user(cursor)
            elif choice == 2:
                delete_user(cursor)
            elif choice == 3:
                partial_username = input("Enter a partial username to search for: ").strip()
                search_users_by_partial_username(cursor, partial_username)
            elif choice == 4:
                update_password(cursor)
            elif choice == 5:
                update_permission_level(cursor)
            elif choice == 6:
                print("Returning to main menu.")
                break
            else:
                print("Invalid choice. Please try again.")

            conn.commit()
            input("Press Enter to return to the user management menu...")

    except sqlite3.Error as e:
        print(f"An error occurred while managing users: {e}")

    finally:
        if 'conn' in locals() and conn:
            conn.close()


def Events_Room_space():
    # Load the event and room data
    event_file_path = 'convention-calendar-2024-10-13_12-42-47.csv'
    events_df = pd.read_csv(event_file_path)
    room_file_path = 'Clark_County_Room_Inventory_Dec_2023.csv'
    hotels_df = pd.read_csv(room_file_path)

    # Data Cleaning and Conversion
    hotels_df['Rooms'] = hotels_df['Rooms'].str.replace(',', '').astype(str)
    hotels_df['Rooms'] = pd.to_numeric(hotels_df['Rooms'], errors='coerce').fillna(0)
    events_df['Start Date'] = pd.to_datetime(events_df['Start Date'], format='%m/%d/%Y')
    events_df['End Date'] = pd.to_datetime(events_df['End Date'], format='%m/%d/%Y')

    # Function to filter events

    def filter_events_dashboard(data, month=None, year=None, venue=None):
        if month:
            data = data[data['Start Date'].dt.month == month]
        if year:
            data = data[data['Start Date'].dt.year == year]
        if venue:
            data = data[data['Venue'].str.contains(venue, case=False, na=False)]
        return data[['Venue', 'Event', 'Start Date', 'End Date', 'Est Attendees']]

    # Function to display the events as a formatted table
    def display_pretty_table(data):
        if data.empty:
            print("No events found for the selected criteria.")
            return
        
        venue_width, event_width, date_width, attendees_width = 50, 60, 12, 15
        header = (f"{'Venue':<{venue_width}} {'Event':<{event_width}} {'Start Date':<{date_width}} "
                f"{'End Date':<{date_width}} {'Est Attendees':<{attendees_width}}")
        print(header)
        print("=" * len(header))
        for _, row in data.iterrows():
            venue = f"{row['Venue']:<{venue_width}}"[:venue_width]
            event = f"{row['Event']:<{event_width}}"[:event_width]
            start_date = row['Start Date'].strftime("%Y-%m-%d")
            end_date = row['End Date'].strftime("%Y-%m-%d")
            attendees = f"{row['Est Attendees']:<{attendees_width}}"
            print(f"{venue} {event} {start_date:<{date_width}} {end_date:<{date_width}} {attendees}")

    # Event Database Menu
    def events_database_menu():
        while True:
            clear_output(wait=True)
            print("\n-- Events Database --")
            print("1. Filter by Month")
            print("2. Filter by Year")
            print("3. Filter by Location (Venue)")
            print("4. View All Events")
            print("5. Go Back")
            display("===================")
            log_access(event_description="User Events Database Menu", log_type="Display- Event Database Menu", user_id=logged_in_user_id)
            choice = input("Select an option: ")
            if choice == '1':
                try:
                    month = int(input("Enter the month (1-12): "))
                    if month < 1 or month > 12:
                        print("Invalid input. Please enter a number between 1 and 12.")
                        continue
                    while True:
                        year = int(input("Enter the year (2024 or 2025): "))
                        if year in [2024, 2025]:
                            break
                        else:
                            print("No events found for the selected year, please try again")
                    print()  # Add a space between user input and the displayed results
                    filtered_events = filter_events_dashboard(events_df, month=month, year=year)
                    display_pretty_table(filtered_events)
                    input("Press enter to continue....")
                except ValueError:
                    print("Invalid input. Please enter a valid month and year.")
            elif choice == '2':
                try:
                    while True:
                        year = int(input("Enter the year (2024 or 2025): "))
                        if year in [2024, 2025]:
                            break
                        else:
                            print("No events found for the selected year, please try again")
                    filtered_events = filter_events_dashboard(events_df, year=year)
                    display_pretty_table(filtered_events)
                    input("Press enter to continue....")
                except ValueError:
                    print("Invalid input. Please enter a valid year.")
            elif choice == '3':
                while True:
                    venue = input("Enter the venue name (e.g., Las Vegas Convention Center): ").strip()
                    filtered_events = filter_events_dashboard(events_df, venue=venue)
                    if not filtered_events.empty:
                        break
                    else:
                        print(f"No events found for the venue: {venue}. Please try again.")
                display_pretty_table(filtered_events)
                input("Press enter to continue....")
            elif choice == '4':
                display_pretty_table(filter_events_dashboard(events_df))
                input("Press enter to continue....")
            elif choice == '5':
                return
            else:
                print("Invalid option. Please try again.")

    # Event and Accommodation Report
    def event_and_accommodation_report():
        while True:
            venue_name = input("Enter the venue name: ")
            matched_hotels = hotels_df[hotels_df['Property Name'].str.contains(venue_name, case=False, na=False, regex=True)]
            if matched_hotels.empty:
                print(f"No rooms found for the venue: {venue_name}")
                continue

            # Display matched hotels and prompt user to select one
            print("\nAvailable Hotels:")
            hotel_names = matched_hotels['Property Name'].tolist()
            available_rooms = matched_hotels['Rooms'].tolist()
            while True:
                for idx, hotel_name in enumerate(hotel_names, start=1):
                    print(f"{idx}. {hotel_name}")

                try:
                    hotel_choice = int(input("Select a hotel by entering the corresponding number: "))
                    if hotel_choice < 1 or hotel_choice > len(hotel_names):
                        print("Invalid choice. Please select a valid hotel number.")
                    else:
                        selected_hotel = hotel_names[hotel_choice - 1]
                        selected_rooms = available_rooms[hotel_choice - 1]
                        print(f"\nYou have selected {selected_hotel} with {selected_rooms} rooms.")
                        break
                except ValueError:
                    print("Invalid input. Please enter a number.")

            # Look up events at the selected hotel
            events_at_hotel = events_df[events_df['Venue'].str.contains(selected_hotel, case=False, na=False, regex=True)]
            if events_at_hotel.empty:
                print(f"No events at this venue: {selected_hotel}")
            else:
                for _, row in events_at_hotel.iterrows():
                    print(f"\nEvent: {row['Event']}")
                    print(f"Venue: {row['Venue']}")
                    print(f"Start Date: {row['Start Date'].strftime('%Y-%m-%d')}")
                    print(f"End Date: {row['End Date'].strftime('%Y-%m-%d')}")
                    print(f"Estimated Attendees: {row['Est Attendees']}")
            break

    # Peak Attendance by Week
    def peak_attendance_scatter_plot():
        events_df['Week'] = events_df['Start Date'].dt.to_period('W')
        weekly_attendees = events_df.groupby('Week')['Est Attendees'].sum().reset_index()
        weekly_attendees['Total Rooms'] = hotels_df['Rooms'].sum()
        weekly_attendees['Week'] = weekly_attendees['Week'].dt.start_time
        x = weekly_attendees['Week']
        y = weekly_attendees['Est Attendees']
        colors = np.random.rand(len(weekly_attendees))
        sizes = 1000 * colors
        plt.figure(figsize=(16, 8))
        plt.scatter(x, y, c=colors, s=sizes, alpha=0.5, cmap='viridis')
        plt.axhline(y=hotels_df['Rooms'].sum(), color='r', linestyle='--', label='Total Available Rooms')
        plt.colorbar(label='Color Scale')
        plt.xlabel('Week')
        plt.ylabel('Estimated Attendees')
        plt.title('Peak Attendance by Week')
        plt.xticks(rotation=45, ha='right')
        plt.legend()
        plt.tight_layout()
        plt.show()

    # Sub Menu
    
    while True:
        clear_output(wait=True)
        print("\n-------------   Events Database   ------------------")
        print(" Las Vegas Travel and Convention Intelligence")
        print(" ---------------------------------------------")
        print("1. Event Data")
        print("2. Event and Accommodation Report")
        print("3. Peak Attendance by Week")
        print("4. Return to Main Menu")
        display("===================")
        log_access(event_description="User Displayed Event Database SubMenu", log_type="Display- Event Database SubMenu", user_id=logged_in_user_id)
        choice = input("Enter your choice: ")
        if choice == '1':
            events_database_menu()
        elif choice == '2':
            event_and_accommodation_report()
            input("Press enter to continue....")
        elif choice == '3':
            peak_attendance_scatter_plot()
            input("Press enter to continue....")
        elif choice == '4':
            
            return

        else:
            print("Invalid option. Please try again.")

def weather_dashboard():
    # Please enter your OpenWeatherMap API Key here
    APPID = '3741775b9523fb5654ad764de205b071'

    # Assume the location to be Las Vegas, NV, US
    location = 'Las Vegas, NV, US'

    # Function to get weather data from OpenWeatherMap API
    def get_weather_data(endpoint, params):
        url = f'https://api.openweathermap.org/data/2.5/{endpoint}'
        params['q'] = location
        params['appid'] = APPID
        params['units'] = 'Imperial'
        response = requests.get(url, params=params)
        response.raise_for_status()
        return json.loads(response.text)

    # Function to display current weather
    def display_current_weather():
        weather_data = get_weather_data('weather', {})
        print('Current weather in', weather_data['name'])
        weather_sec = weather_data['weather'][0]['description']
        main_sec = weather_data['main']
        print(f"Description: {weather_sec}")
        print(f"Temperature: {main_sec['temp']}\u00b0F")
        print(f"Feels like: {main_sec['feels_like']}\u00b0F")
        print(f"Humidity: {main_sec['humidity']}%")
        input("Press enter to continue....")

    # Function to display and summarize forecast weather
    def display_forecast_weather():
        forecast_data = get_weather_data('forecast', {'cnt': 40})  # 5-day forecast
        dates = [item['dt_txt'] for item in forecast_data['list']]
        temperatures = [item['main']['temp'] for item in forecast_data['list']]
        humidities = [item['main']['humidity'] for item in forecast_data['list']]
        wind_speeds = [item['wind']['speed'] for item in forecast_data['list']]

        forecast_df = pd.DataFrame({
            'Date': pd.to_datetime(dates),
            'Temperature (\u00b0F)': temperatures,
            'Humidity (%)': humidities,
            'Wind Speed (mph)': wind_speeds
        })

        # Summarize the forecast weather
        print("\nSummary of Forecast Weather for Las Vegas, NV, US:")
        print(f"Average Temperature: {forecast_df['Temperature (\u00b0F)'].mean():.2f}\u00b0F")
        print(f"Average Humidity: {forecast_df['Humidity (%)'].mean():.2f}%")
        print(f"Average Wind Speed: {forecast_df['Wind Speed (mph)'].mean():.2f} mph")
        print(f"Temperature Range: {forecast_df['Temperature (\u00b0F)'].min():.2f}\u00b0F - {forecast_df['Temperature (\u00b0F)'].max():.2f}\u00b0F")
        print(f"Humidity Range: {forecast_df['Humidity (%)'].min():.2f}% - {forecast_df['Humidity (%)'].max():.2f}%")
        print(f"Wind Speed Range: {forecast_df['Wind Speed (mph)'].min():.2f} mph - {forecast_df['Wind Speed (mph)'].max():.2f} mph")

        # Visualization
        plt.figure(figsize=(10, 6))
        plt.plot(forecast_df['Date'], forecast_df['Temperature (\u00b0F)'], label='Temperature (\u00b0F)', color='orange')
        plt.title('Forecasted Temperature Over Time')
        plt.xlabel('Date')
        plt.ylabel('Temperature (\u00b0F)')
        plt.legend()
        plt.grid()
        plt.show()

        sns.histplot(forecast_df['Humidity (%)'], kde=True, bins=20, color='blue')
        plt.title('Distribution of Forecasted Humidity')
        plt.xlabel('Humidity (%)')
        plt.ylabel('Frequency')
        plt.show()

        sns.boxplot(data=forecast_df[['Temperature (\u00b0F)', 'Humidity (%)', 'Wind Speed (mph)']])
        plt.title('Boxplot of Forecast Data')
        plt.xticks([0, 1, 2], ['Temperature (\u00b0F)', 'Humidity (%)', 'Wind Speed (mph)'])
        plt.ylabel('Values')
        plt.show()
        input("Press enter to continue....")

    # Function to display and summarize monthly weather (using the 5-day forecast as an estimate)
    def display_monthly_weather():
        # Load weather data
        weather_report = pd.read_csv("LV-WeatherReport.csv")
        
        # Convert 'Date' column to datetime
        weather_report['Date'] = pd.to_datetime(weather_report['Date'], format='%m/%d/%y')
        
        # Extract Month and Year from 'Date'
        weather_report['Month'] = weather_report['Date'].dt.month  # Month
        weather_report['Year'] = weather_report['Date'].dt.year    # Year
        # Replace 0 values in 'Min Temperature' with the median value for that month to maintain the pattern
        weather_report['Min Temperature'] = weather_report.groupby('Month')['Min Temperature'].transform(
            lambda x: x.replace(0, x.median()))
        
        # Group data by Year and Month, calculate max, min, mean temperatures and total precipitation
        monthly_stats = weather_report.groupby(['Year', 'Month']).agg(
            maxTemp=('Max Temperature', 'max'),   # Max temperature of the month
            minTemp=('Min Temperature', 'min'),   # Min temperature of the month
            meanTemp=('Avg Temperature', 'mean'), # Average temperature of the month
            totalPrecipitation=('Precipitation (in)', 'sum')  # Total precipitation of the month
        ).reset_index()
        
        # Ask user for the year to display
        print('Please enter the year to review data (from 2019 to 2024):')
        year = input()
        
        # Check if the entered year is valid
        while not year.isdigit() or int(year) not in monthly_stats['Year'].values:
            print('Currently, only the data from 2019 to 2024 are available. Please enter again:')
            year = input()
        
        # Filter data by the selected year
        data = monthly_stats[monthly_stats['Year'] == int(year)]
        
        # Plotting
        plt.figure(figsize=(12, 8))  # Set figure size

        # Create a single plot with lines for max, min, and mean temperatures
        sns.lineplot(data=data, x='Month', y='maxTemp', marker='o', label='Max Temp', color='red')
        sns.lineplot(data=data, x='Month', y='minTemp', marker='o', label='Min Temp', color='blue')
        sns.lineplot(data=data, x='Month', y='meanTemp', marker='o', label='Mean Temp', color='green')
        
        # Create bar chart for total precipitation
        plt.bar(data['Month'], data['totalPrecipitation'], label='Total Precipitation (inches)', color='lightblue', alpha=0.5)
        
        # Customize the plot
        plt.title(f"Las Vegas Temperature and Precipitation Data for {year}")
        plt.xlabel("Month", fontsize=12)
        plt.ylabel("Temperature (°F), Precipitation (in)")
        plt.xticks(ticks=range(1, 13), labels=["Jan", "Feb", "Mar", "Apr", "May", "Jun", "Jul", "Aug", "Sep", "Oct", "Nov", "Dec"])
        plt.grid(True)
        plt.legend()
        
        # Show the plot
        plt.tight_layout()
        plt.show()
        input("Press enter to continue....")

    
    while True:
        clear_output(wait=True)
        print("\nWeather Dashboard Main Menu")
        print("1. Current Weather")
        print("2. Forecast Weather")
        print("3. Monthly Weather")
        print("4. Return to Main Menu")
        display("===================")
        log_access(event_description="User Displayed Weather Dashboard Menu", log_type="Display- Weather Dashboard Menu", user_id=logged_in_user_id)
        menu_input = input("Please enter menu selection (1 - 4): ")

        if menu_input == "1":
            display_current_weather()
        elif menu_input == "2":
            display_forecast_weather()
        elif menu_input == "3":
            display_monthly_weather()
        elif menu_input == "4":
            return
        else:
            print("Invalid choice. Please try again.")

def airport_menu():
    while True:
        clear_output(wait=True)
        print("\n-- Airport Statistics --")
        print("1. Airport Statistics Yearly ")
        print("2. Return to Main Menu")
        display("===================")
        log_access(event_description="User Airport Stats Menu", log_type="Display- Airport Stats Menu", user_id=logged_in_user_id)
        choice = input("Select an option: ")
        # read the csv file for the data
        passengers = pd.read_csv('LV-Passengers.csv')
        flights = pd.read_csv('LV-Flights.csv')
        #Combine all data into one DataFrame
        transportation = pd.merge(passengers, flights, how = 'inner', on = ['Year','Month'])
        transportation
            
        # Data Cleaning:
        #Filter the TOTAL row 
        transportation = transportation.loc[transportation['Month'] != 'TOTAL']
        #Drop TOTAL PASSENGERS & TOTAL FLIGHTS column
        transportation = transportation.drop(['TOTAL PASSENGERS', 'TOTAL FLIGHTS'], axis = 1)
        #Remove commas and convert columns to numeric
        #Replace the NaN with median within a year
        dataCleaning = ['DOMESTIC PASSENGERS', 'INTERNATIONAL PASSENGERS', 
                            'DOMESTIC FLIGHTS', 'INTERNATIONAL FLIGHTS']
        for col in dataCleaning:
            transportation[col] = transportation[col].replace(',', '', regex=True).astype(float)
            transportation[col] = transportation.groupby('Year')[col].transform(lambda x: x.fillna(x.median()))
        if choice == '1':
            #Ask user to enter the year
            print('Please enter the year to review data (from 2002 to 2024)')
            year = input()
            
            #Verify the year in the DataFrame
            while not year.isdigit() or int(year) not in transportation['Year'].values:
                print('Currently, only the data form 2002 to 2024 are available, please enter again:')
                year = input()
            
            #Create the mask for filtering by year
            mask = transportation['Year'] == int(year)
            data = transportation[mask]
            data
            maskHistory = transportation['Year'] == int(year) - 1
            dataHistory = transportation[maskHistory]
            
            # Ensure both datasets have the same months
            uniqueMonths = data['Month'].unique()
            
            # Reindex both data and dataHistory to match months
            data = data.set_index('Month').reindex(uniqueMonths).reset_index()
            dataHistory = dataHistory.set_index('Month').reindex(uniqueMonths).reset_index()
            
            # Fill missing values in dataHistory with zeros
            dataHistory = dataHistory.fillna(0)
            
            # Side-by-side bar chart to compare current year with previous year
            if not dataHistory.empty:  # Check if there is data for the previous year
                plt.figure(figsize=(10, 6))
                width = 0.4  # Set Bar width
                x = range(len(data['Month']))
                # Plot the input year data
                plt.bar(data['Month'], data['DOMESTIC PASSENGERS'], width = width, label='Domestic '+str(year))
                plt.bar(data['Month'], data['INTERNATIONAL PASSENGERS'], width = width, 
                        bottom=data['DOMESTIC PASSENGERS'], label='International '+str(year))
                
                # Offset x-axis for the previous year
                xHistory = [i - width for i in x]
                
                # Plot the previous year data
                plt.bar(xHistory, dataHistory['DOMESTIC PASSENGERS'], width = width, label='Domestic '+str(int(year) - 1))
                plt.bar(xHistory, dataHistory['INTERNATIONAL PASSENGERS'], width = width, 
                        bottom=dataHistory['DOMESTIC PASSENGERS'], label='International '+str(int(year) - 1))
            
            else:
                # Create stacked bar chart for passengers
                plt.figure(figsize=(6, 5))
                plt.bar(data['Month'], data['DOMESTIC PASSENGERS'], label='Domestic '+str(year))
                plt.bar(data['Month'], data['INTERNATIONAL PASSENGERS'], 
                        bottom=data['DOMESTIC PASSENGERS'], label='International '+str(year))
            
            # Customize the plot
            plt.title("Domestic and International Passengers ("+str(int(year) - 1)+" and "+str(year)+")")
            plt.xlabel("Month")
            plt.ylabel("Number of Passengers (millions)")
            plt.legend(loc='lower right')
            plt.show()

            # Side-by-side bar chart to compare current year with previous year
            if not dataHistory.empty:  # Check if there is data for the previous year
                plt.figure(figsize=(10, 6))
                width = 0.4  # Bar width
                x = range(len(data['Month']))
                # Plot current year data
                plt.bar(data['Month'], data['DOMESTIC FLIGHTS'], width = width, label='Domestic '+str(year))
                plt.bar(data['Month'], data['INTERNATIONAL FLIGHTS'], width = width, 
                        bottom=data['DOMESTIC FLIGHTS'], label='International '+str(year))
                
                # Offset x-axis for the previous year
                x_history = [i - width for i in x]
                
                # Plot previous year data
                plt.bar(x_history, dataHistory['DOMESTIC FLIGHTS'], width = width, label='Domestic '+str(int(year) - 1))
                plt.bar(x_history, dataHistory['INTERNATIONAL FLIGHTS'], width = width, 
                        bottom=dataHistory['DOMESTIC FLIGHTS'], label='International '+str(int(year) - 1))
            else:
                # Create stacked bar chart for passengers
                plt.figure(figsize=(6, 5))
                plt.bar(data['Month'], data['DOMESTIC FLIGHTS'], label='Domestic '+str(year))
                plt.bar(data['Month'], data['INTERNATIONAL FLIGHTS'], 
                        bottom=data['DOMESTIC FLIGHTS'], label='International '+str(year))
                
            # Customize the plot
            plt.title("Domestic and International Flights ("+str(int(year) - 1)+" and "+str(year)+")")
            plt.xlabel("Month")
            plt.ylabel("Number of Flights")
            plt.legend(loc='lower right')
            plt.show()

            # Calculate total domestic and international passengers for the selected year
            totalDomestic = data['DOMESTIC PASSENGERS'].sum()
            totalInternational = data['INTERNATIONAL PASSENGERS'].sum()
            
            # Create data for the pie chart
            categories = ['Domestic Passengers', 'International Passengers']
            sizes = [totalDomestic, totalInternational]
            plt.figure(figsize=(5, 5))
            plt.pie(sizes, labels=categories, autopct='%1.2f%%')
            plt.title("Proportion of Domestic vs International Passengers ("+str(year)+")")
            plt.show()

            input("Press enter to continue....")
            
        elif choice == '2':
            return
        else:
            print("Invalid option. Please try again.")


def meeting_space():
    # read csv files and load into dataframe
    df2 = pd.read_csv('Meeting_Space.csv')
    # Change the display options
    pd.set_option('display.max_rows', 500)
    pd.set_option('display.max_columns', 15)
    
    #Change column type
    df2 = df2.drop('Property Count', axis=1)
    df2.index.set_names('Property', inplace=True)
    # Add columns to datafram
    df2['City'] = 'Las Vegas'
    # Add a state column with value set to NV
    df2['State'] = 'NV'
    # convert dataframe columns to integer and address comma seperator
    df2['Hotel Room Inventory'] = df2['Hotel Room Inventory'].str.replace(',', '').astype(int)
    df2['Exhibit Meeting Area'] = df2['Exhibit Meeting Area'].str.replace(',', '').astype(int)
    df2['Hotel Room Inventory'] = df2['Hotel Room Inventory'].fillna('0')
    df2['Exhibit Meeting Area'] = df2['Exhibit Meeting Area'].fillna('0')  
    
    while True:
        clear_output(wait=True)
        print("\n-- Meeting Space Inventory --")
        print("1. Meeting Space Calculator and Recommendation ")
        print("2. Return to Main Menu")
        display("===================")
        log_access(event_description="User Displayed Meeting Space Menu", log_type="Display- Meeting Space Menu", user_id=logged_in_user_id)
        choice = input("Select an option: ")
    
        if choice == '1':

            # Meeting Space Calculator and Recommendation Engine for Las Vegas Events
            print('Meeting Space Calculator and Recommendation Engine for Las Vegas Events\n')
                
            # Function to get a valid integer input
            def get_valid_integer(prompt):
                while True:
                    user_input = input(prompt)
                    # Check if the input is a digit
                    if user_input.isdigit() and int(user_input) > 0:  
                        return int(user_input)  # Convert to integer and return
                    else:
                        print("Invalid input. Please enter a valid number.")
                
            # Get number of attendees
            attendees = get_valid_integer('Please enter the estimated number of attendees at your event: ')
                
            # Get number of hotel rooms required
            rooms = get_valid_integer('Please enter the number of hotel rooms required for your event: ')
                
            # Get minimum space required
            space_min = get_valid_integer('Please enter the minimum space required (sq. ft) for your meeting: ')
                
            # Get maximum space required
            space_max = get_valid_integer('Please enter the maximum space required (sq. ft) for your meeting: ')
                
            # Display the inputs back to the user
            print("\nEvent Details:")
            print(f"Estimated Attendees: {attendees}")
            print(f"Hotel Rooms Required: {rooms}")
            print(f"Meeting Space Required: {space_min} - {space_max} sq. ft")
                
            # Create boolean mask with user input
            mask_area = (df2['Exhibit Meeting Area'] >= int(space_min)) & (df2['Exhibit Meeting Area'] <= int(space_max))
            mask_room = (df2['Hotel Room Inventory'] >= int(rooms))
                
            # Show results of boolean masking on user input
            df3 = df2.loc[mask_area]
            df4 = df3.loc[mask_room]
            if df4.empty:
                print("\nNo venues match your criteria.")
            else:
                print('----------------------- Vegas IQ ---------------------------\n')
                print('The following meeting venues match your search requirements\n')
                print('Hotel rooms required:                  '+str(rooms))
                print('Exhibit space min (sq.ft.) required:   '+str(space_min))
                print('Exhibit space max (sq.ft.) required:   '+str(space_max))
                df_final = df4[['Property', 'City', 'Exhibit Meeting Area', 'Hotel Room Inventory']]
                print(df_final.to_markdown(index=False))

            input("Press enter to continue....")
        elif choice == '2':
            return
        else:
            print("Invalid option. Please try again.")



def main():    
    """
    Main program loop for VegasIQ with integrated user authentication, dynamic menu,
    and global user tracking.
    """
    initialize_database_once()  # Ensure the database is initialized only once

    # Authenticate user before accessing the main menu
    permission_level = None
    while permission_level is None:
        clear_output(wait=True)
        display("")
        permission_level = authenticate_user()


    while True:
        clear_output(wait=True)
        choice = display_main_menu(permission_level)
        if choice == 1:
            Events_Room_space()
        elif choice == 2:
            weather_dashboard()
        elif choice == 3:
            airport_menu()
        elif choice == 4:
            meeting_space()
        elif choice == 5 and permission_level == 2:
            user_management()
        elif choice == 6:
            clear_output(wait=True)
            print("Thank you for using VegasIQ!")
            break
        else:
            print("Invalid choice. Please try again.")
            input("Press Enter to return to the main menu...")


main()


User Management Menu
1. Create a New User
2. Delete a User
3. Search for Users by Partial Username
4. Update Password by Partial Username Search
5. Update Permission Level by Partial Username Search
6. Return to Main Menu


''

Select an option:  2


Select a user to delete (enter the number, or 0 to go back):  1
Are you sure you want to delete user jorge.alvarez17? (yes/no):  yes


User deleted successfully.
Database error: database is locked


Authentication successful!


In [ ]:
1